# TanggapMakna — SRL Feasibility V2

## Mengapa V2 dibuat?

Smoke test V1 berhasil membuktikan bahwa:
- dataset dapat dibaca;
- split tidak bocor;
- tokenizer/model berjalan;
- GPU dan training pipeline bekerja.

Namun model V1 **collapse ke label `O`** (`F1 = 0`) karena formulasi eksperimen perlu diperbaiki.

### Koreksi utama V2
1. **Tidak menggunakan BIO-SRL 29 kelas.** Dataset asli mendefinisikan **15 semantic-role labels langsung per token**, sehingga kolom `argument` digunakan sebagai target apa adanya.
2. Menggunakan **`indolem/indobert-base-uncased`**, yaitu IndoBERT BERT-Base yang sesuai dengan deskripsi model pada paper pendamping dataset.
3. Mempertahankan **official held-out test set** dataset: 250 teks per jenis kejadian = 1.000 test text.
4. Sisa 3.150 teks digunakan sebagai train/validation.
5. Evaluasi menggunakan **token-level Precision, Recall, F1**, dengan perhatian khusus pada **Macro-F1 non-O**, bukan accuracy saja.
6. Training mengikuti hyperparameter paper sedekat mungkin: LR `2e-5`, batch `16`, weight decay `0.01`, seed `42`, maksimal `10 epoch` + early stopping.

> V2 masih supervised baseline. Semi-supervised pseudo-labeling dari paper **belum** diterapkan karena tujuan kita adalah membuktikan model dasar yang layak untuk produk TanggapMakna terlebih dahulu.

## 1. Setup GPU dan library

In [ ]:
# ============================================================
# TANGGAPMAKNA — SRL FEASIBILITY V2
# CELL 1: SAFE COLAB ENVIRONMENT SETUP
# ============================================================

# ------------------------------------------------------------
# 1. Cek GPU
# ------------------------------------------------------------

!nvidia-smi


# ------------------------------------------------------------
# 2. Install HANYA package tambahan yang kita perlukan
#
# PENTING:
# - Jangan upgrade pandas
# - Jangan upgrade requests
# - Jangan upgrade paket inti Google Colab
# - Tanpa -U agar dependency bawaan Colab dipertahankan
# ------------------------------------------------------------

!pip install -q \
    transformers \
    datasets \
    accelerate \
    iterative-stratification


# ------------------------------------------------------------
# 3. Cek dependency environment
# ------------------------------------------------------------

!pip check


# ------------------------------------------------------------
# 4. Import
# ------------------------------------------------------------

import os
import re
import json
import random
import shutil
import zipfile

from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

from sklearn.preprocessing import MultiLabelBinarizer

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

from iterstrat.ml_stratifiers import (
    MultilabelStratifiedShuffleSplit,
)

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)


# ------------------------------------------------------------
# 5. Reproducibility
# ------------------------------------------------------------

SEED = 42

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)


# ------------------------------------------------------------
# 6. Environment information
# ------------------------------------------------------------

print()
print("=" * 72)
print("TANGGAPMAKNA — SRL FEASIBILITY V2")
print("=" * 72)

print("Python environment ready.")
print()

print("PyTorch version     :", torch.__version__)
print("Pandas version      :", pd.__version__)

import transformers
import datasets
import accelerate
import sklearn

print("Transformers version:", transformers.__version__)
print("Datasets version    :", datasets.__version__)
print("Accelerate version  :", accelerate.__version__)
print("Scikit-learn version:", sklearn.__version__)

print()
print("CUDA available      :", torch.cuda.is_available())


if torch.cuda.is_available():

    print(
        "GPU                 :",
        torch.cuda.get_device_name(0),
    )

    print(
        "CUDA version        :",
        torch.version.cuda,
    )

else:

    raise RuntimeError(
        "GPU belum aktif.\n"
        "Buka Runtime > Change runtime type > pilih GPU."
    )


# ------------------------------------------------------------
# 7. Sanity check Pandas
# ------------------------------------------------------------

test_df = pd.DataFrame(
    {
        "text": [
            "banjir di lampung",
            "gempa di jawa",
        ],
        "label": [
            "FLOOD",
            "EARTHQUAKE",
        ],
    }
)

assert len(test_df) == 2

print()
print("Pandas sanity check : OK")
print("Environment         : READY")

print("=" * 72)

Wed Aug 12 04:17:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Upload dataset

Upload:

`Semantic Role Labeling Datasets for Crisis Event.zip`

In [ ]:
from google.colab import files

uploaded = files.upload()

zip_candidates = [
    Path("/content") / name
    for name in uploaded.keys()
    if name.lower().endswith(".zip")
]

if not zip_candidates:
    raise RuntimeError("ZIP dataset tidak ditemukan.")

ZIP_PATH = zip_candidates[0]

print("ZIP:", ZIP_PATH)
print("Ukuran:", f"{ZIP_PATH.stat().st_size/(1024**2):.2f} MB")

Saving Semantic Role Labeling Datasets for Crisis Event.zip to Semantic Role Labeling Datasets for Crisis Event.zip
ZIP: /content/Semantic Role Labeling Datasets for Crisis Event.zip
Ukuran: 66.71 MB


## 3. Ekstrak CSV Bahasa Indonesia

In [ ]:
WORK_DIR = Path("/content/tanggapmakna_v2")
WORK_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    candidates = [
        n for n in z.namelist()
        if n.endswith("3-Labeled Data/all-labeled-data.csv")
        and "Version 2.0" not in n
    ]

    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected 1 Indonesian labeled CSV, ditemukan {len(candidates)}."
        )

    MEMBER = candidates[0]
    z.extract(MEMBER, WORK_DIR)

CSV_PATH = WORK_DIR / MEMBER

df = pd.read_csv(CSV_PATH)

print("Shape          :", df.shape)
print("Columns        :", list(df.columns))
print("Unique text_id :", df["text_id"].nunique())
print("Blank token    :", df["token"].isna().sum())

display(df.head(15))

Shape          : (99206, 5)
Columns        : ['text_id', 'id', 'token', 'entity', 'argument']
Unique text_id : 4150
Blank token    : 4


,text_id,id,token,entity,argument
0,BAN-00001,BAN-00001.001,13,O,O
1,BAN-00001,BAN-00001.002,janji,O,O
2,BAN-00001,BAN-00001.003,akan,O,O
3,BAN-00001,BAN-00001.004,mencari,O,O
4,BAN-00001,BAN-00001.005,solusi,O,O
5,BAN-00001,BAN-00001.006,banjir,B-EVE,FALSE-EVENT
6,BAN-00001,BAN-00001.007,jakarta,B-LOC,PLACE-ARG
7,BAN-00001,BAN-00001.008,dan,O,O
8,BAN-00001,BAN-00001.009,kemacetan,O,O
9,BAN-00001,BAN-00001.010,jakarta,B-LOC,PLACE-ARG


## 4. Cleaning minimal + validasi 15 label SRL

In [ ]:
df_clean = (
    df[df["token"].notna()]
    .copy()
    .reset_index(drop=True)
)

EXPECTED_LABELS = {
    "O",
    "PLACE-ARG",
    "FALSE-EVENT",
    "STREET-ARG",
    "AFFECTEDOBJECTS-ARG",
    "TIME-ARG",
    "OFFICER-ARG",
    "INFORMATION-ARG",
    "FIRE-EVENT",
    "REASON-ARG",
    "ACCIDENT-EVENT",
    "DEATHVICTIM-ARG",
    "EARTHQUAKE-EVENT",
    "FLOOD-EVENT",
    "WOUNDVICTIM-ARG",
}

actual_labels = set(df_clean["argument"].unique())

print("Jumlah label:", len(actual_labels))
print("Labels:")
for x in sorted(actual_labels):
    print(" -", x)

if actual_labels != EXPECTED_LABELS:
    print("\nMissing:", EXPECTED_LABELS - actual_labels)
    print("Extra  :", actual_labels - EXPECTED_LABELS)
    raise RuntimeError("Label dataset tidak sesuai ekspektasi.")

print("\n✅ 15 semantic-role labels terverifikasi.")

display(
    df_clean["argument"]
    .value_counts()
    .to_frame("jumlah_token")
)

Jumlah label: 15
Labels:
 - ACCIDENT-EVENT
 - AFFECTEDOBJECTS-ARG
 - DEATHVICTIM-ARG
 - EARTHQUAKE-EVENT
 - FALSE-EVENT
 - FIRE-EVENT
 - FLOOD-EVENT
 - INFORMATION-ARG
 - O
 - OFFICER-ARG
 - PLACE-ARG
 - REASON-ARG
 - STREET-ARG
 - TIME-ARG
 - WOUNDVICTIM-ARG

✅ 15 semantic-role labels terverifikasi.


,jumlah_token
argument,
O,68751
PLACE-ARG,10227
FALSE-EVENT,3448
STREET-ARG,3154
AFFECTEDOBJECTS-ARG,2789
TIME-ARG,2424
OFFICER-ARG,2113
INFORMATION-ARG,1977
FIRE-EVENT,834


## 5. Rekonstruksi official test set

Paper pendamping menyebut:
- **3.150 labeled training texts**
- **1.000 held-out testing texts**
- test dibuat seimbang: **250 teks untuk masing-masing** banjir, gempa, kebakaran, kecelakaan.

Pada file dataset, 250 teks pertama (`001–250`) dari setiap prefix membentuk tepat 1.000 teks tersebut. Sisa data membentuk tepat 3.150 teks.


In [ ]:
def text_number(text_id):
    return int(str(text_id).split("-")[1])

def event_prefix(text_id):
    return str(text_id).split("-")[0]

all_text_ids = (
    df_clean["text_id"]
    .drop_duplicates()
    .tolist()
)

official_test_ids = {
    tid for tid in all_text_ids
    if text_number(tid) <= 250
}

official_train_pool_ids = {
    tid for tid in all_text_ids
    if tid not in official_test_ids
}

print("Train pool :", len(official_train_pool_ids))
print("Test       :", len(official_test_ids))

assert len(official_train_pool_ids) == 3150
assert len(official_test_ids) == 1000
assert official_train_pool_ids.isdisjoint(official_test_ids)

print("\nDistribusi TEST:")
test_id_series = pd.Series(sorted(official_test_ids))
display(
    test_id_series
    .str[:3]
    .value_counts()
    .rename_axis("event")
    .to_frame("jumlah_teks")
)

assert all(
    test_id_series.str[:3].value_counts().get(p, 0) == 250
    for p in ["BAN", "GEM", "KEB", "KEC"]
)

print("✅ Official test = 250 teks per event.")

Train pool : 3150
Test       : 1000

Distribusi TEST:


,jumlah_teks
event,
BAN,250
GEM,250
KEB,250
KEC,250


✅ Official test = 250 teks per event.


## 6. Cek proporsi label pada official train/test

In [ ]:
df_train_pool_tokens = df_clean[
    df_clean["text_id"].isin(official_train_pool_ids)
]

df_test_tokens = df_clean[
    df_clean["text_id"].isin(official_test_ids)
]

def label_distribution(frame):
    c = frame["argument"].value_counts()
    pct = frame["argument"].value_counts(normalize=True) * 100
    return pd.DataFrame({
        "tokens": c,
        "percent": pct.round(3),
    })

print("TRAIN POOL")
display(label_distribution(df_train_pool_tokens))

print("\nTEST")
display(label_distribution(df_test_tokens))

print(
    "\nO ratio train:",
    round((df_train_pool_tokens["argument"] == "O").mean() * 100, 2),
    "%"
)

print(
    "O ratio test :",
    round((df_test_tokens["argument"] == "O").mean() * 100, 2),
    "%"
)

TRAIN POOL


,tokens,percent
argument,,
O,54201,70.138
PLACE-ARG,7912,10.238
FALSE-EVENT,2647,3.425
STREET-ARG,2643,3.420
AFFECTEDOBJECTS-ARG,1995,2.582
TIME-ARG,1777,2.299
OFFICER-ARG,1703,2.204
INFORMATION-ARG,1403,1.816
FIRE-EVENT,641,0.829



TEST


,tokens,percent
argument,,
O,14550,66.366
PLACE-ARG,2315,10.559
FALSE-EVENT,801,3.654
AFFECTEDOBJECTS-ARG,794,3.622
TIME-ARG,647,2.951
INFORMATION-ARG,574,2.618
STREET-ARG,511,2.331
OFFICER-ARG,410,1.870
REASON-ARG,256,1.168



O ratio train: 70.14 %
O ratio test : 66.37 %


## 7. Bentuk satu record per `text_id` — target = `argument` langsung

In [ ]:
records = []

for text_id, group in df_clean.groupby("text_id", sort=False):
    tokens = group["token"].astype(str).tolist()
    labels = group["argument"].astype(str).tolist()

    assert len(tokens) == len(labels)

    records.append({
        "text_id": text_id,
        "event_prefix": event_prefix(text_id),
        "tokens": tokens,
        "labels": labels,
    })

records_df = pd.DataFrame(records)

print("Jumlah text :", len(records_df))
print("Mean token  :", records_df["tokens"].map(len).mean())
print("Max token   :", records_df["tokens"].map(len).max())

display(records_df.head())

Jumlah text : 4150
Mean token  : 23.904096385542168
Max token   : 126


,text_id,event_prefix,tokens,labels
0,BAN-00001,BAN,"[13, janji, akan, mencari, solusi, banjir, jak...","[O, O, O, O, O, FALSE-EVENT, PLACE-ARG, O, O, ..."
1,BAN-00002,BAN,"[ingin, tenangkan, diri, di, musim, banjir, ?,...","[O, O, O, O, O, FALSE-EVENT, O, O, O, O, O, O,..."
2,BAN-00003,BAN,"[intensitas, dan, curah, hujan, yang, tinggi, ...","[REASON-ARG, REASON-ARG, REASON-ARG, REASON-AR..."
3,BAN-00004,BAN,"[ayuh, kita, bersama, menjayakan, misi, bantua...","[O, O, O, O, O, O, O, FALSE-EVENT, O, PLACE-AR..."
4,BAN-00005,BAN,"[anak, buah, bantu, set, up, pusat, pemindahan...","[O, O, O, O, O, O, O, FALSE-EVENT, O, O, O, O,..."


## 8. Split 3.150 train pool menjadi train/validation 80/20

Official test tetap utuh.

Validation dipilih dengan multilabel stratification berdasarkan role yang muncul pada tiap teks.

In [ ]:
train_pool_df = records_df[
    records_df["text_id"].isin(official_train_pool_ids)
].reset_index(drop=True)

test_records_df = records_df[
    records_df["text_id"].isin(official_test_ids)
].reset_index(drop=True)

train_pool_df["role_set"] = train_pool_df["labels"].apply(
    lambda labs: sorted(set(labs) - {"O"})
)

mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(train_pool_df["role_set"])
X = np.arange(len(train_pool_df)).reshape(-1, 1)

splitter = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED,
)

train_idx, val_idx = next(splitter.split(X, Y))

train_records_df = train_pool_df.iloc[train_idx].reset_index(drop=True)
val_records_df = train_pool_df.iloc[val_idx].reset_index(drop=True)

print("Train      :", len(train_records_df))
print("Validation :", len(val_records_df))
print("Test       :", len(test_records_df))

train_ids = set(train_records_df["text_id"])
val_ids = set(val_records_df["text_id"])
test_ids = set(test_records_df["text_id"])

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)

print("✅ Tidak ada data leakage.")

Train      : 2518
Validation : 632
Test       : 1000
✅ Tidak ada data leakage.


## 9. Distribusi role per split

In [ ]:
def role_presence_counts(frame):
    counter = Counter()
    for labels in frame["labels"]:
        counter.update(set(labels) - {"O"})
    return counter

rows = []

for split_name, frame in [
    ("train", train_records_df),
    ("validation", val_records_df),
    ("test", test_records_df),
]:
    counts = role_presence_counts(frame)

    for label in sorted(EXPECTED_LABELS - {"O"}):
        rows.append({
            "split": split_name,
            "label": label,
            "texts_with_label": counts.get(label, 0),
        })

role_split = pd.DataFrame(rows)

pivot = role_split.pivot(
    index="label",
    columns="split",
    values="texts_with_label",
).fillna(0).astype(int)

display(pivot)

if (pivot == 0).any().any():
    print("⚠️ Ada label yang hilang dari salah satu split.")
else:
    print("✅ Semua non-O role muncul di seluruh split.")

split,test,train,validation
label,,,
ACCIDENT-EVENT,177,324,81
AFFECTEDOBJECTS-ARG,310,654,164
DEATHVICTIM-ARG,57,126,31
EARTHQUAKE-EVENT,83,129,32
FALSE-EVENT,472,1447,362
FIRE-EVENT,145,407,102
FLOOD-EVENT,119,202,50
INFORMATION-ARG,139,281,70
OFFICER-ARG,201,619,155


✅ Semua non-O role muncul di seluruh split.


## 10. Label mapping

In [ ]:
label_list = ["O"] + sorted(EXPECTED_LABELS - {"O"})

label2id = {
    label: i
    for i, label in enumerate(label_list)
}

id2label = {
    i: label
    for label, i in label2id.items()
}

print("Jumlah kelas:", len(label_list))
print(json.dumps(label2id, indent=2, ensure_ascii=False))

Jumlah kelas: 15
{
  "O": 0,
  "ACCIDENT-EVENT": 1,
  "AFFECTEDOBJECTS-ARG": 2,
  "DEATHVICTIM-ARG": 3,
  "EARTHQUAKE-EVENT": 4,
  "FALSE-EVENT": 5,
  "FIRE-EVENT": 6,
  "FLOOD-EVENT": 7,
  "INFORMATION-ARG": 8,
  "OFFICER-ARG": 9,
  "PLACE-ARG": 10,
  "REASON-ARG": 11,
  "STREET-ARG": 12,
  "TIME-ARG": 13,
  "WOUNDVICTIM-ARG": 14
}


## 11. Load model utama: IndoBERT Base (IndoLEM)

Model:
`indolem/indobert-base-uncased`

Alasan:
- BERT Base khusus Bahasa Indonesia;
- 12 layer, hidden size 768, 12 attention heads;
- pretraining >220 juta kata Bahasa Indonesia;
- sesuai dengan model IndoBERT yang dideskripsikan pada penelitian SRL krisis pendamping dataset.

In [ ]:
MODEL_NAME = "indolem/indobert-base-uncased"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer :", tokenizer.__class__.__name__)
print("Fast      :", getattr(tokenizer, "is_fast", False))
print("Vocab     :", len(tokenizer))

config.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/234k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer : BertTokenizer
Fast      : True
Vocab     : 31923


## 12. Encode token → WordPiece

Label diberikan hanya pada **subword pertama** dari token asli.
Subword lanjutan diberi `-100`, sehingga tidak ikut dihitung dalam loss/evaluasi.

In [ ]:
def make_dataset(frame):
    return Dataset.from_dict({
        "text_id": frame["text_id"].tolist(),
        "tokens": frame["tokens"].tolist(),
        "labels_str": frame["labels"].tolist(),
    })

train_ds = make_dataset(train_records_df)
val_ds = make_dataset(val_records_df)
test_ds = make_dataset(test_records_df)


def encode_words_and_labels(example):
    input_ids = [tokenizer.cls_token_id]
    label_ids = [-100]

    for word, label in zip(
        example["tokens"],
        example["labels_str"],
    ):
        pieces = tokenizer.tokenize(str(word))

        if not pieces:
            pieces = [tokenizer.unk_token]

        piece_ids = tokenizer.convert_tokens_to_ids(pieces)

        remaining = MAX_LENGTH - 1 - len(input_ids)

        if remaining <= 0:
            break

        piece_ids = piece_ids[:remaining]

        input_ids.extend(piece_ids)

        # Hanya subword pertama memakai target label.
        label_ids.append(label2id[label])
        label_ids.extend([-100] * (len(piece_ids) - 1))

    input_ids.append(tokenizer.sep_token_id)
    label_ids.append(-100)

    attention_mask = [1] * len(input_ids)

    assert len(input_ids) == len(label_ids)
    assert len(input_ids) == len(attention_mask)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": label_ids,
    }


train_encoded = train_ds.map(
    encode_words_and_labels,
    remove_columns=train_ds.column_names,
)

val_encoded = val_ds.map(
    encode_words_and_labels,
    remove_columns=val_ds.column_names,
)

test_encoded = test_ds.map(
    encode_words_and_labels,
    remove_columns=test_ds.column_names,
)

print(train_encoded)
print(val_encoded)
print(test_encoded)

Map:   0%|          | 0/2518 [00:00<?, ? examples/s]

Map:   0%|          | 0/632 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2518
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 632
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1000
})


## 13. Audit alignment GOLD sebelum training

In [ ]:
def alignment_audit(frame, encoded_ds, n=5):
    sample_indices = np.linspace(
        0,
        len(frame)-1,
        min(n, len(frame)),
        dtype=int,
    )

    for idx in sample_indices:
        original = frame.iloc[idx]
        encoded = encoded_ds[int(idx)]

        print("\n" + "=" * 90)
        print("TEXT ID:", original["text_id"])
        print("TEXT   :", " ".join(original["tokens"]))

        rows = []
        cursor = 1  # CLS

        for word, gold_label in zip(
            original["tokens"],
            original["labels"],
        ):
            pieces = tokenizer.tokenize(str(word))
            if not pieces:
                pieces = [tokenizer.unk_token]

            if cursor >= len(encoded["labels"]) - 1:
                break

            encoded_label_id = encoded["labels"][cursor]

            encoded_label = (
                id2label[encoded_label_id]
                if encoded_label_id != -100
                else "IGNORED"
            )

            rows.append({
                "word": word,
                "wordpieces": " | ".join(pieces),
                "gold_original": gold_label,
                "gold_encoded": encoded_label,
                "match": gold_label == encoded_label,
            })

            cursor += len(pieces)

        audit_df = pd.DataFrame(rows)
        display(audit_df)

        assert audit_df["match"].all(), (
            f"Alignment gagal pada {original['text_id']}"
        )


alignment_audit(
    train_records_df,
    train_encoded,
    n=5,
)

print("✅ Gold-label alignment valid.")


TEXT ID: BAN-01126
TEXT   : tweeps jika ada banjir di wilayahmu laporkan ke selain terdata akan segera ada penanganan di level kelurahan ini sudah sop dari bpbd akan percuma kalau kamu mention saya bilang 'ci banjir nih ! sudah salah mention kagak ada informasinya pula


,word,wordpieces,gold_original,gold_encoded,match
0,tweeps,twe | ##eps,O,O,True
1,jika,jika,O,O,True
2,ada,ada,O,O,True
3,banjir,banjir,FALSE-EVENT,FALSE-EVENT,True
4,di,di,O,O,True
5,wilayahmu,wilayah | ##mu,O,O,True
6,laporkan,lapor | ##kan,O,O,True
7,ke,ke,O,O,True
8,selain,selain,O,O,True
9,terdata,terd | ##ata,O,O,True



TEXT ID: BAN-01891
TEXT   : banjir di konawe sultra membuat biawak2 cina ini keluar dari persembunyian eanaknya di apai ya -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -


,word,wordpieces,gold_original,gold_encoded,match
0,banjir,banjir,FALSE-EVENT,FALSE-EVENT,True
1,di,di,O,O,True
2,konawe,konawe,PLACE-ARG,PLACE-ARG,True
3,sultra,sul | ##tra,PLACE-ARG,PLACE-ARG,True
4,membuat,membuat,O,O,True
5,biawak2,bia | ##wak | ##2,O,O,True
6,cina,cina,PLACE-ARG,PLACE-ARG,True
7,ini,ini,O,O,True
8,keluar,keluar,O,O,True
9,dari,dari,O,O,True



TEXT ID: KEB-00398
TEXT   : jabar faktajabar telah terjadi kebakaran dini hari tadi selasa 12 mei 2020 sekitar pukul 0 . di daerah campaka depan rs bhakti husada ii purwakarta pt putra naga indotama . . . bagi warga purwakarta yg tau info


...


TEXT ID: KEC-01432
TEXT   : kembali terjadi kecelakaan kerja di jict 02/07/18 dini hari bukti kalo direktur operasional dan jajarannya tidak mampu dan punya kapabilitas dalam menjaga keamanan dan keselamatan kerja zero accident hanya pencitraan


,word,wordpieces,gold_original,gold_encoded,match
0,kembali,kembali,O,O,True
1,terjadi,terjadi,O,O,True
2,kecelakaan,kecelakaan,FALSE-EVENT,FALSE-EVENT,True
3,kerja,kerja,O,O,True
4,di,di,O,O,True
5,jict,ji | ##c | ##t,O,O,True
6,02/07/18,02 | / | 07 | / | 18,TIME-ARG,TIME-ARG,True
7,dini,dini,O,O,True
8,hari,hari,O,O,True
9,bukti,bukti,O,O,True



TEXT ID: KEC-01124
TEXT   : 550 : : kecelakaan di cikarang timur km 40 arah cawang kendaraan truk box terbalik miring di lajur kiri dalam penanganan petugas


,word,wordpieces,gold_original,gold_encoded,match
0,550,550,O,O,True
1,:,:,O,O,True
2,:,:,O,O,True
3,kecelakaan,kecelakaan,ACCIDENT-EVENT,ACCIDENT-EVENT,True
4,di,di,O,O,True
5,cikarang,cikarang,PLACE-ARG,PLACE-ARG,True
6,timur,timur,PLACE-ARG,PLACE-ARG,True
7,km,km,STREET-ARG,STREET-ARG,True
8,40,40,STREET-ARG,STREET-ARG,True
9,arah,arah,STREET-ARG,STREET-ARG,True


✅ Gold-label alignment valid.


## 14. Cek truncation

In [ ]:
def encoded_length(tokens):
    total = 2

    for word in tokens:
        pieces = tokenizer.tokenize(str(word))
        total += max(1, len(pieces))

    return total


all_lengths = records_df["tokens"].apply(encoded_length)

print(all_lengths.describe())

too_long = int((all_lengths > MAX_LENGTH).sum())

print(
    f"Jumlah > {MAX_LENGTH}:",
    too_long,
    "/",
    len(all_lengths),
)

if too_long > 0:
    print("⚠️ Ada input terpotong. Perlu ditinjau.")
else:
    print("✅ Tidak ada truncation.")

count    4150.000000
mean       30.395422
std        15.024252
min         6.000000
25%        18.000000
50%        28.000000
75%        42.000000
max       186.000000
Name: tokens, dtype: float64
Jumlah > 256: 0 / 4150
✅ Tidak ada truncation.


## 15. Model + metrik token-level

Metrik utama:
- Accuracy seluruh token
- **Macro-F1 non-O**
- Micro-F1 non-O
- Weighted-F1 non-O
- Precision/Recall non-O

`O` tidak dihapus dari proses training. Ia hanya tidak dijadikan pusat penilaian F1 karena dominan.

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    label_pad_token_id=-100,
)

NON_O_IDS = [
    label2id[x]
    for x in label_list
    if x != "O"
]


def flatten_eval(logits, labels):
    pred_ids = np.argmax(logits, axis=-1)

    y_true = []
    y_pred = []

    for pred_row, gold_row in zip(pred_ids, labels):
        for pred_id, gold_id in zip(pred_row, gold_row):
            if gold_id == -100:
                continue

            y_true.append(int(gold_id))
            y_pred.append(int(pred_id))

    return np.array(y_true), np.array(y_pred)


def compute_metrics(eval_pred):
    logits, labels = eval_pred

    y_true, y_pred = flatten_eval(logits, labels)

    accuracy = accuracy_score(y_true, y_pred)

    p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=NON_O_IDS,
        average="macro",
        zero_division=0,
    )

    p_micro, r_micro, f_micro, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=NON_O_IDS,
        average="micro",
        zero_division=0,
    )

    _, _, f_weighted, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=NON_O_IDS,
        average="weighted",
        zero_division=0,
    )

    predicted_non_o_rate = float(
        np.mean(y_pred != label2id["O"])
    )

    gold_non_o_rate = float(
        np.mean(y_true != label2id["O"])
    )

    return {
        "accuracy": accuracy,
        "precision_macro_no_o": p_macro,
        "recall_macro_no_o": r_macro,
        "macro_f1_no_o": f_macro,
        "micro_f1_no_o": f_micro,
        "weighted_f1_no_o": f_weighted,
        "predicted_non_o_rate": predicted_non_o_rate,
        "gold_non_o_rate": gold_non_o_rate,
    }


print("Model:", model.__class__.__name__)
print("Parameters:", f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M")

[transformers] You passed `num_labels=15` which is incompatible to the `id2label` map of length `2`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  445MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: indolem/indobert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those param

Model: BertForTokenClassification
Parameters: 110.0M


## 16. Training IndoBERT Base

Konfigurasi mengikuti paper pendamping sedekat mungkin:
- learning rate = `2e-5`
- batch size = `16`
- max epoch = `10`
- weight decay = `0.01`
- seed = `42`
- early stopping

**Test set tidak digunakan untuk memilih model.**

In [ ]:
OUTPUT_DIR = "/content/tanggapmakna_v2_indobert"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1_no_o",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_encoded,
    eval_dataset=val_encoded,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ],
)

train_result = trainer.train()

print("\n" + "=" * 72)
print("TRAINING SELESAI")
print("=" * 72)
print(train_result)

model.safetensors: reconstructing file:   0%|          |  0.00B /  445MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro No O,Recall Macro No O,Macro F1 No O,Micro F1 No O,Weighted F1 No O,Predicted Non O Rate,Gold Non O Rate
1,0.421642,0.324417,0.899950,0.702130,0.631589,0.617008,0.786906,0.766769,0.295814,0.291667
2,0.264293,0.270774,0.910885,0.724235,0.733480,0.706453,0.815040,0.807635,0.306687,0.291667
3,0.205095,0.262405,0.915410,0.727122,0.747316,0.729481,0.821973,0.818849,0.302099,0.291667
4,0.181013,0.270918,0.912205,0.723760,0.810330,0.761393,0.828665,0.832119,0.323089,0.291667
5,0.138190,0.266913,0.923077,0.781691,0.782627,0.778400,0.843017,0.840957,0.303230,0.291667
6,0.117685,0.278929,0.921317,0.755294,0.796007,0.772185,0.840297,0.839132,0.308446,0.291667
7,0.100112,0.296427,0.917924,0.747223,0.803389,0.770882,0.837568,0.837306,0.319256,0.291667


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


TRAINING SELESAI
TrainOutput(global_step=1106, training_loss=0.24831277681733557, metrics={'train_runtime': 331.3459, 'train_samples_per_second': 75.993, 'train_steps_per_second': 4.768, 'total_flos': 537405645609000.0, 'train_loss': 0.24831277681733557, 'epoch': 7.0})


## 17. Validation dan held-out TEST

In [ ]:
val_metrics = trainer.evaluate(
    val_encoded,
    metric_key_prefix="val",
)

test_metrics = trainer.evaluate(
    test_encoded,
    metric_key_prefix="test",
)

print("VALIDATION")
print(json.dumps(val_metrics, indent=2))

print("\nHELD-OUT TEST")
print(json.dumps(test_metrics, indent=2))

[transformers] early stopping required metric_for_best_model, but did not find eval_macro_f1_no_o so early stopping is disabled


Training Loss,Validation Loss,Epoch,Accuracy,Precision Macro No O,Recall Macro No O,Macro F1 No O,Micro F1 No O,Weighted F1 No O,Predicted Non O Rate,Gold Non O Rate
0.100112,0.266913,7,0.923077,0.781691,0.782627,0.778400,0.843017,0.840957,0.303230,0.291667


[transformers] early stopping required metric_for_best_model, but did not find eval_macro_f1_no_o so early stopping is disabled


Training Loss,Validation Loss,Epoch,Accuracy,Precision Macro No O,Recall Macro No O,Macro F1 No O,Micro F1 No O,Weighted F1 No O,Predicted Non O Rate,Gold Non O Rate
0.100112,0.552459,7,0.862479,0.747610,0.657470,0.677730,0.741289,0.741063,0.335203,0.336344


VALIDATION
{
  "val_loss": 0.2669132947921753,
  "val_accuracy": 0.9230769230769231,
  "val_precision_macro_no_o": 0.7816913783016062,
  "val_recall_macro_no_o": 0.7826265717423825,
  "val_macro_f1_no_o": 0.7784001766946936,
  "val_micro_f1_no_o": 0.8430171138812592,
  "val_weighted_f1_no_o": 0.8409573565183053,
  "val_predicted_non_o_rate": 0.30323026646556056,
  "val_gold_non_o_rate": 0.2916666666666667
}

HELD-OUT TEST
{
  "test_loss": 0.552458643913269,
  "test_accuracy": 0.8624794745484401,
  "test_precision_macro_no_o": 0.7476103847037565,
  "test_recall_macro_no_o": 0.6574696470149105,
  "test_macro_f1_no_o": 0.6777298258163258,
  "test_micro_f1_no_o": 0.7412891394416898,
  "test_weighted_f1_no_o": 0.7410634699328686,
  "test_predicted_non_o_rate": 0.3352034300310162,
  "test_gold_non_o_rate": 0.33634373289545705
}


## 18. Classification report test — non-O labels

In [ ]:
pred_output = trainer.predict(test_encoded)

y_true, y_pred = flatten_eval(
    pred_output.predictions,
    pred_output.label_ids,
)

non_o_names = [
    id2label[i]
    for i in NON_O_IDS
]

report = classification_report(
    y_true,
    y_pred,
    labels=NON_O_IDS,
    target_names=non_o_names,
    digits=4,
    zero_division=0,
)

print(report)

                     precision    recall  f1-score   support

     ACCIDENT-EVENT     0.8627    0.7822    0.8205       225
AFFECTEDOBJECTS-ARG     0.8272    0.7594    0.7919       794
    DEATHVICTIM-ARG     0.7625    0.6667    0.7114       183
   EARTHQUAKE-EVENT     0.6993    0.4714    0.5632       227
        FALSE-EVENT     0.7592    0.7715    0.7653       801
         FIRE-EVENT     0.8632    0.8497    0.8564       193
        FLOOD-EVENT     0.8806    0.4184    0.5673       141
    INFORMATION-ARG     0.5086    0.4617    0.4840       574
        OFFICER-ARG     0.6557    0.7756    0.7106       410
          PLACE-ARG     0.8479    0.8402    0.8440      2315
         REASON-ARG     0.8072    0.2617    0.3953       256
         STREET-ARG     0.4393    0.8845    0.5870       511
           TIME-ARG     0.9451    0.7975    0.8650       647
    WOUNDVICTIM-ARG     0.6081    0.4639    0.5263        97

          micro avg     0.7426    0.7400    0.7413      7374
          macro avg   

## 19. Predicted-label distribution — cek collapse

In [ ]:
pred_counter = Counter(
    id2label[int(i)]
    for i in y_pred
)

gold_counter = Counter(
    id2label[int(i)]
    for i in y_true
)

dist_rows = []

for label in label_list:
    dist_rows.append({
        "label": label,
        "gold_tokens": gold_counter.get(label, 0),
        "predicted_tokens": pred_counter.get(label, 0),
    })

display(
    pd.DataFrame(dist_rows)
    .sort_values("gold_tokens", ascending=False)
)

if pred_counter.get("O", 0) == len(y_pred):
    print("❌ COLLAPSE: semua prediksi O.")
else:
    print("✅ Model menghasilkan label non-O.")

,label,gold_tokens,predicted_tokens
0,O,14550,14575
10,PLACE-ARG,2315,2294
5,FALSE-EVENT,801,814
2,AFFECTEDOBJECTS-ARG,794,729
13,TIME-ARG,647,546
8,INFORMATION-ARG,574,521
12,STREET-ARG,511,1029
9,OFFICER-ARG,410,485
11,REASON-ARG,256,83
4,EARTHQUAKE-EVENT,227,153


✅ Model menghasilkan label non-O.


## 20. Error analysis sampel held-out test

In [ ]:
def predict_word_labels(tokens):
    dummy = {
        "tokens": tokens,
        "labels_str": ["O"] * len(tokens),
    }

    encoded = encode_words_and_labels(dummy)

    inputs = {
        "input_ids": torch.tensor(
            [encoded["input_ids"]],
            device=model.device,
        ),
        "attention_mask": torch.tensor(
            [encoded["attention_mask"]],
            device=model.device,
        ),
    }

    with torch.no_grad():
        logits = model(**inputs).logits[0]

    pred_ids = logits.argmax(dim=-1).cpu().tolist()

    result = []
    cursor = 1

    for word in tokens:
        pieces = tokenizer.tokenize(str(word))
        if not pieces:
            pieces = [tokenizer.unk_token]

        if cursor >= len(pred_ids) - 1:
            break

        result.append(id2label[int(pred_ids[cursor])])
        cursor += len(pieces)

    return result


sample = test_records_df.sample(
    n=10,
    random_state=SEED,
)

for _, row in sample.iterrows():
    tokens = row["tokens"]
    gold = row["labels"]
    pred = predict_word_labels(tokens)

    n = min(len(tokens), len(gold), len(pred))

    print("\n" + "=" * 95)
    print("TEXT ID:", row["text_id"])
    print("TEXT   :", " ".join(tokens))

    rows = []

    for i in range(n):
        if gold[i] != "O" or pred[i] != "O":
            rows.append({
                "token": tokens[i],
                "gold": gold[i],
                "pred": pred[i],
                "benar": gold[i] == pred[i],
            })

    display(pd.DataFrame(rows))


TEXT ID: KEC-00022
TEXT   : klik kecelakaan mobil di km 93 cipali arah jakarta lalu lintas macet


,token,gold,pred,benar
0,kecelakaan,ACCIDENT-EVENT,ACCIDENT-EVENT,True
1,mobil,AFFECTEDOBJECTS-ARG,AFFECTEDOBJECTS-ARG,True
2,km,O,STREET-ARG,False
3,93,O,STREET-ARG,False
4,cipali,PLACE-ARG,STREET-ARG,False
5,arah,O,STREET-ARG,False
6,jakarta,PLACE-ARG,STREET-ARG,False
7,lalu,INFORMATION-ARG,INFORMATION-ARG,True
8,lintas,INFORMATION-ARG,INFORMATION-ARG,True
9,macet,INFORMATION-ARG,INFORMATION-ARG,True



TEXT ID: KEC-00238
TEXT   : hati-hati kecelakaan di pucuk lamongan


,token,gold,pred,benar
0,kecelakaan,FALSE-EVENT,FALSE-EVENT,True
1,pucuk,PLACE-ARG,PLACE-ARG,True
2,lamongan,PLACE-ARG,PLACE-ARG,True



TEXT ID: BAN-00126
TEXT   : artikel tentang banjir berita hoaks di tengah pemberitaan media yang semakin partisan


...

,token,gold,pred,benar
0,kecelakaan,FALSE-EVENT,FALSE-EVENT,True
1,balap,REASON-ARG,O,False
2,moto,REASON-ARG,O,False
3,gp,REASON-ARG,O,False



TEXT ID: BAN-00245
TEXT   : | potensi hujan lebat meningkat waspada banjir dan longsor di beberapa wilayah indonesia * slh 1 nya jogja 7 - 10 maret 2018 via


,token,gold,pred,benar
0,hujan,REASON-ARG,O,False
1,lebat,REASON-ARG,O,False
2,meningkat,REASON-ARG,O,False
3,banjir,FALSE-EVENT,FALSE-EVENT,True
4,indonesia,PLACE-ARG,PLACE-ARG,True
5,jogja,PLACE-ARG,PLACE-ARG,True
6,10,TIME-ARG,O,False
7,maret,TIME-ARG,O,False
8,2018,TIME-ARG,O,False



TEXT ID: GEM-00012
TEXT   : gempa 4 6 sr guncang simeulue aceh : gempa bumi berkekuatan 4 6 skala richter terjadi di


,token,gold,pred,benar
0,gempa,EARTHQUAKE-EVENT,EARTHQUAKE-EVENT,True
1,4,INFORMATION-ARG,INFORMATION-ARG,True
2,6,INFORMATION-ARG,INFORMATION-ARG,True
3,sr,INFORMATION-ARG,INFORMATION-ARG,True
4,guncang,EARTHQUAKE-EVENT,O,False
5,simeulue,PLACE-ARG,PLACE-ARG,True
6,aceh,PLACE-ARG,PLACE-ARG,True
7,gempa,EARTHQUAKE-EVENT,EARTHQUAKE-EVENT,True
8,bumi,EARTHQUAKE-EVENT,EARTHQUAKE-EVENT,True
9,4,INFORMATION-ARG,INFORMATION-ARG,True


## 21. Inference laporan baru

Karena dataset tidak memakai BIO, span untuk UI dibuat dengan menggabungkan token berurutan yang memiliki semantic role sama.

In [ ]:
def simple_word_tokenize(text):
    return re.findall(
        r"\w+(?:[-']\w+)*|[^\w\s]",
        text.lower(),
        flags=re.UNICODE,
    )


def labels_to_spans(tokens, labels):
    spans = []
    current_role = None
    current_tokens = []

    def flush():
        nonlocal current_role, current_tokens

        if current_role and current_tokens:
            spans.append({
                "role": current_role,
                "text": " ".join(current_tokens),
            })

        current_role = None
        current_tokens = []

    for token, label in zip(tokens, labels):
        if label == "O":
            flush()
            continue

        if label != current_role:
            flush()
            current_role = label
            current_tokens = [token]
        else:
            current_tokens.append(token)

    flush()

    return spans


def predict_report(text):
    tokens = simple_word_tokenize(text)
    labels = predict_word_labels(tokens)

    tokens = tokens[:len(labels)]

    spans = labels_to_spans(
        tokens,
        labels,
    )

    print("\nINPUT:")
    print(text)

    display(
        pd.DataFrame({
            "token": tokens,
            "pred": labels,
        })
    )

    print("EXTRACTED:")
    if spans:
        display(pd.DataFrame(spans))
    else:
        print("(tidak ada role terdeteksi)")

    return {
        "tokens": tokens,
        "labels": labels,
        "spans": spans,
    }


examples = [
    "Banjir terjadi di Sukarame dan merendam beberapa rumah.",
    "Kebakaran melanda pasar di Bandar Lampung pada malam hari.",
    "Gempa mengguncang wilayah Lampung sekitar pukul 14.30.",
    "Kecelakaan terjadi di Jalan Soekarno Hatta dan menyebabkan dua orang terluka.",
    "Pemerintah sedang membahas solusi banjir Jakarta.",
]

results_new = []

for text in examples:
    print("\n" + "#" * 95)
    results_new.append(
        predict_report(text)
    )


###############################################################################################

INPUT:
Banjir terjadi di Sukarame dan merendam beberapa rumah.


,token,pred
0,banjir,FLOOD-EVENT
1,terjadi,O
2,di,O
3,sukarame,PLACE-ARG
4,dan,O
5,merendam,INFORMATION-ARG
6,beberapa,AFFECTEDOBJECTS-ARG
7,rumah,AFFECTEDOBJECTS-ARG
8,.,O


EXTRACTED:


,role,text
0,FLOOD-EVENT,banjir
1,PLACE-ARG,sukarame
2,INFORMATION-ARG,merendam
3,AFFECTEDOBJECTS-ARG,beberapa rumah



###############################################################################################

INPUT:
Kebakaran melanda pasar di Bandar Lampung pada malam hari.


...

,role,text
0,ACCIDENT-EVENT,kecelakaan
1,STREET-ARG,jalan soekarno hatta
2,WOUNDVICTIM-ARG,dua
3,DEATHVICTIM-ARG,orang
4,WOUNDVICTIM-ARG,terluka



###############################################################################################

INPUT:
Pemerintah sedang membahas solusi banjir Jakarta.


,token,pred
0,pemerintah,O
1,sedang,O
2,membahas,O
3,solusi,O
4,banjir,FALSE-EVENT
5,jakarta,PLACE-ARG
6,.,O


EXTRACTED:


,role,text
0,FALSE-EVENT,banjir
1,PLACE-ARG,jakarta


## 22. Prototype Completeness Engine

In [ ]:
EVENT_ROLES = {
    "FLOOD-EVENT",
    "EARTHQUAKE-EVENT",
    "FIRE-EVENT",
    "ACCIDENT-EVENT",
}

LOCATION_ROLES = {
    "PLACE-ARG",
    "STREET-ARG",
}

IMPACT_ROLES = {
    "AFFECTEDOBJECTS-ARG",
    "INFORMATION-ARG",
}

TIME_ROLES = {
    "TIME-ARG",
}

VICTIM_ROLES = {
    "DEATHVICTIM-ARG",
    "WOUNDVICTIM-ARG",
}


def completeness_from_spans(spans):
    roles = {
        item["role"]
        for item in spans
    }

    status = {
        "event": bool(roles & EVENT_ROLES),
        "location": bool(roles & LOCATION_ROLES),
        "impact": bool(roles & IMPACT_ROLES),
        "time": bool(roles & TIME_ROLES),
        "victim": bool(roles & VICTIM_ROLES),
    }

    missing_questions = []

    if not status["event"]:
        missing_questions.append(
            "Jenis kejadian apa yang sedang dilaporkan?"
        )

    if not status["location"]:
        missing_questions.append(
            "Di mana lokasi kejadian?"
        )

    if not status["impact"]:
        missing_questions.append(
            "Apa dampak yang terlihat atau objek apa yang terdampak?"
        )

    if not status["time"]:
        missing_questions.append(
            "Kapan kejadian atau kondisi ini diamati?"
        )

    if not status["victim"]:
        missing_questions.append(
            "Apakah terdapat korban? Jika belum diketahui, pilih 'Tidak tahu'."
        )

    return {
        "status": status,
        "completeness_score": sum(status.values()) / len(status),
        "missing_questions": missing_questions,
    }


demo = predict_report(
    "Banjir terjadi di Sukarame dekat pasar."
)

completeness = completeness_from_spans(
    demo["spans"]
)

print(
    json.dumps(
        completeness,
        indent=2,
        ensure_ascii=False,
    )
)


INPUT:
Banjir terjadi di Sukarame dekat pasar.


,token,pred
0,banjir,FLOOD-EVENT
1,terjadi,O
2,di,O
3,sukarame,PLACE-ARG
4,dekat,INFORMATION-ARG
5,pasar,INFORMATION-ARG
6,.,O


EXTRACTED:


,role,text
0,FLOOD-EVENT,banjir
1,PLACE-ARG,sukarame
2,INFORMATION-ARG,dekat pasar


{
  "status": {
    "event": true,
    "location": true,
    "impact": true,
    "time": false,
    "victim": false
  },
  "completeness_score": 0.6,
  "missing_questions": [
    "Kapan kejadian atau kondisi ini diamati?",
    "Apakah terdapat korban? Jika belum diketahui, pilih 'Tidak tahu'."
  ]
}


## 23. Simpan model dan manifest eksperimen

In [ ]:
EXPORT_DIR = Path(
    "/content/TanggapMakna_IndoBERT_V2"
)
EXPORT_DIR.mkdir(exist_ok=True)

trainer.save_model(EXPORT_DIR)
tokenizer.save_pretrained(EXPORT_DIR)

metadata = {
    "model_name": MODEL_NAME,
    "seed": SEED,
    "max_length": MAX_LENGTH,
    "labels": label_list,
    "official_train_pool_size": len(official_train_pool_ids),
    "validation_size": len(val_records_df),
    "final_train_size": len(train_records_df),
    "official_test_size": len(official_test_ids),
    "test_metrics": test_metrics,
}

(EXPORT_DIR / "experiment_metadata.json").write_text(
    json.dumps(
        metadata,
        indent=2,
        ensure_ascii=False,
        default=float,
    ),
    encoding="utf-8",
)

(EXPORT_DIR / "train_ids.json").write_text(
    json.dumps(sorted(train_ids), indent=2),
    encoding="utf-8",
)

(EXPORT_DIR / "validation_ids.json").write_text(
    json.dumps(sorted(val_ids), indent=2),
    encoding="utf-8",
)

(EXPORT_DIR / "test_ids.json").write_text(
    json.dumps(sorted(test_ids), indent=2),
    encoding="utf-8",
)

archive = shutil.make_archive(
    "/content/TanggapMakna_IndoBERT_V2",
    "zip",
    root_dir=EXPORT_DIR,
)

print("Saved:", archive)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved: /content/TanggapMakna_IndoBERT_V2.zip
